Setup environment

In [1]:
# Install required packages for local ADK execution and Agent Platform deployment
!pip install --upgrade --quiet "google-cloud-aiplatform[agent_engines,adk]>=1.112" google-adk google-genai requests geopy

import os


PROJECT_ID = "qwiklabs-gcp-04-c28c99d58f4c"
LOCATION = "us-central1"

os.environ["GOOGLE_GENAI_USE_ENTERPRISE"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

print(f"✅ Environment configured for Project: {PROJECT_ID} ({LOCATION})")

✅ Environment configured for Project: qwiklabs-gcp-04-c28c99d58f4c (us-central1)


Logging and Input Validation

In [8]:
import time
from datetime import datetime
from google import genai
from google.genai import types
from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmRequest, LlmResponse

LOG_FILE = "fema_agent_interactions.log"

def write_log_entry(message: str):
    """Cloud-safe logger: prints to standard output (for Cloud Logging) and appends to local file safely."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    entry = f"{timestamp} [INFO] {message}"
    print(entry)
    try:
        with open(LOG_FILE, "a", encoding="utf-8") as f:
            f.write(entry + "\n")
    except Exception:
        pass

def validate_fema_input(user_input: str, history_context: str = "") -> tuple[bool, str]:
    """
    Validates that user input is strictly appropriate and related to FEMA's mission,
    catching both direct off-topic requests and mid-conversation 'topic drift' (e.g., tourism).
    """
    refusal_msg = (
        "I am a FEMA Emergency Response Assistant. I can only assist with weather alerts, "
        "disaster news, evacuation routes, and emergency preparedness."
    )

    lower_in = user_input.lower()

    # 1. Expanded Fast-Pass Blocklist (catches recipes, coding, weapons, and tourism/sightseeing phrases)
    off_topic_keywords = [
        "recipe", "sourdough", "cake", "lasagna", "chili", "stock market", "scrape",
        "movie", "football", "basketball", "video game", "homework", "bomb", "weapon",
        "sightseeing", "tourist", "tourism", "things to do", "fun things", "restaurants",
        "bars", "nightlife", "attractions", "museums", "vacation", "entertainment"
    ]
    if any(k in lower_in for k in off_topic_keywords):
        return True, refusal_msg

    # 2. Strict LLM Guardrail Prompt that explicitly forbids tourism/leisure follow-ups
    prompt = f"""
    You are a strict security and mission-scope input validator for a FEMA Emergency Response Assistant.

    ALLOWED TOPICS ONLY:
    - US weather forecasts, severe weather, and active alerts
    - Breaking disaster/emergency news and situational awareness
    - Evacuation routes, highway directions to safety, and travel readiness
    - Emergency preparedness, safety checklists, shelter info, and FEMA capabilities

    STRICTLY FORBIDDEN (Must classify as OFF_TOPIC, even if the city was mentioned earlier in an evacuation context):
    - Sightseeing, tourism, leisure activities, dining, restaurants, entertainment, or "what to do for fun / what to do while in [City]"
    - Cooking/recipes, coding/programming, finance/stocks, sports, trivia
    - Weapons, explosives, self-harm, or illegal acts

    CRITICAL RULE ON FOLLOW-UPS:
    - Use `Recent Conversation Context` ONLY to resolve pronouns or missing locations for valid emergency questions (e.g., "What is the weather tomorrow?" or "What should I pack in my car for the drive?").
    - If the user pivots from an evacuation into asking what to do in the destination city (unless they explicitly ask about emergency shelters or safety protocols), you MUST classify it as OFF_TOPIC.

    Recent Conversation Context:
    {history_context or "None"}

    Current User Input: "{user_input}"
    Reply with ONLY one word: VALID or OFF_TOPIC.
    """
    try:
        local_client = genai.Client()
        resp = local_client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt,
            config=types.GenerateContentConfig(temperature=0.0)
        )
        verdict = (resp.text or "").strip().upper()
        if "OFF_TOPIC" in verdict or "VALID" not in verdict:
            return True, refusal_msg
    except Exception as e:
        write_log_entry(f"[GUARDRAIL WARNING]: {e}")
    return False, ""

def before_model_callback_handler(callback_context: CallbackContext, llm_request: LlmRequest) -> LlmResponse | None:
    """
    ADK Callback:
    1. Logs every interaction before the model runs.
    2. Maintains rolling multi-turn memory in `callback_context.state["chat_history"]`.
    3. Validates user input with conversation context and injects context into the workflow nodes.
    """
    agent_name = callback_context.agent_name
    last_text = ""
    last_role = ""
    if llm_request.contents and llm_request.contents[-1].parts:
        last_role = llm_request.contents[-1].role
        for part in llm_request.contents[-1].parts:
            if hasattr(part, "text") and part.text:
                last_text = part.text
                break

    write_log_entry(f"[BEFORE_MODEL - {agent_name}] ({last_role}): {last_text[:120] or '[Tool/System Event]'}")

    # Retrieve rolling conversation history from ADK session state
    history_list = callback_context.state.get("chat_history", [])
    history_str = "\n".join(history_list[-6:])  # Keep last 6 entries (3 user/assistant turns)

    # When a new user message arrives at the root coordinator:
    if agent_name == "fema_emergency_coordinator" and last_role == "user" and last_text:
        write_log_entry(f"[USER INPUT]: {last_text}")

        # 1. Validate with conversation history
        is_blocked, refusal_text = validate_fema_input(last_text, history_context=history_str)
        if is_blocked:
            write_log_entry(f"[GUARDRAIL BLOCKED]: {refusal_text}")
            return LlmResponse(
                content=types.Content(
                    role="model",
                    parts=[types.Part.from_text(text=refusal_text)]
                )
            )

        # 2. Record the raw user question in session state
        callback_context.state["last_user_query"] = last_text
        history_list.append(f"User: {last_text}")
        callback_context.state["chat_history"] = history_list

        # 3. Prepend rolling conversation context to the user message so the Coordinator remembers prior locations/topics
        if len(history_list) > 1:
             contextualized_text = (
                 f"[CONVERSATION HISTORY FOR CONTEXT:\n{history_str}]\n\n"
                 f"CURRENT USER REQUEST: {last_text}\n"
                 f"(Note: If the user does not specify a city/location, use the location mentioned in the Conversation History above.)"
             )
             llm_request.contents[-1].parts[0].text = contextualized_text

    # Also give the Validator node visibility into the user's question & history so it doesn't strip out context
    elif agent_name == "response_validator" and last_text:
        user_q = callback_context.state.get("last_user_query", "")
        llm_request.contents[-1].parts[0].text = (
            f"Conversation Context:\n{history_str}\n\n"
            f"User Question: {user_q}\n\n"
            f"Coordinator Draft Response to Validate:\n{last_text}"
        )

    return None

def after_model_callback_handler(callback_context: CallbackContext, llm_response: LlmResponse) -> LlmResponse | None:
    """ADK Callback: Logs all model outputs and saves the final refined response into rolling session memory."""
    agent_name = callback_context.agent_name
    out_summary = ""
    full_text = ""
    if llm_response.content and llm_response.content.parts:
        for part in llm_response.content.parts:
            if hasattr(part, "function_call") and part.function_call:
                out_summary = f"Tool Call -> {part.function_call.name}"
                break
            elif hasattr(part, "text") and part.text:
                full_text = part.text
                out_summary = full_text.replace("\n", " ")[:120]

    write_log_entry(f"[AFTER_MODEL - {agent_name}]: {out_summary}...")

    # Save the final refined answer into rolling session state so the next turn remembers what was said
    if agent_name == "response_refiner" and full_text:
        history_list = callback_context.state.get("chat_history", [])
        # Store a concise summary of the assistant's reply in history to keep token count small
        history_list.append(f"FEMA Assistant: {full_text[:350]}...")
        callback_context.state["chat_history"] = history_list

    return None

write_log_entry("=== FEMA Context-Aware Interaction Logger Initialized ===")

2026-10-27 19:15:47 [INFO] === FEMA Context-Aware Interaction Logger Initialized ===


Set up Tools (Weather, News Search, Routing)

In [9]:
import os
import urllib.parse
import requests
from geopy.geocoders import Nominatim
from google import genai
from google.genai import types

def get_nws_weather(city: str) -> str:
    """
    Retrieves current US weather forecasts and active NWS severe weather alerts for a US city.
    Args:
        city (str): US city and state, e.g. "Tampa, FL" or "Easton, WA".
    """
    try:
        geolocator = Nominatim(user_agent="fema_poc_weather_tool")
        location = geolocator.geocode(city, timeout=10)
        if not location:
            return f"Error: Could not find coordinates for {city}."

        lat, lon = round(location.latitude, 4), round(location.longitude, 4)
        headers = {"User-Agent": "(fema-emergency-poc, student@example.com)"}

        points_resp = requests.get(f"https://api.weather.gov/points/{lat},{lon}", headers=headers, timeout=10).json()
        if "properties" not in points_resp:
            return f"Error: {city} appears to be outside the United States. NWS only covers US territories."

        forecast_url = points_resp["properties"]["forecast"]
        periods = requests.get(forecast_url, headers=headers, timeout=10).json()["properties"]["periods"]

        alerts_resp = requests.get(f"https://api.weather.gov/alerts/active?point={lat},{lon}", headers=headers, timeout=10).json()
        alerts = alerts_resp.get("features", [])

        out = [f"=== NWS Weather & Alerts for {city} ({lat}, {lon}) ==="]
        if alerts:
            out.append("🚨 ACTIVE ALERTS:")
            for a in alerts[:3]:
                p = a.get("properties", {})
                out.append(f"- {p.get('event')}: {p.get('headline')}")
        else:
            out.append("✅ No active severe weather alerts.")

        out.append("📅 FORECAST:")
        for period in periods[:4]:  # Top 4 periods (2 days) keeps token usage low!
            out.append(f"- {period['name']}: {period['detailedForecast']}")
        return "\n".join(out)
    except Exception as e:
        return f"Weather lookup error: {str(e)}"


def search_disaster_news(query: str) -> str:
    """
    Searches the internet via Google Search Grounding for real-time disaster news, alerts, and preparedness info.
    Args:
        query (str): The disaster, news, or preparedness topic to search.
    """
    try:
        local_client = genai.Client()
        response = local_client.models.generate_content(
            model="gemini-2.5-flash",
            contents=f"Provide concise, factual current emergency/disaster news or FEMA guidance for: {query}",
            config=types.GenerateContentConfig(
                tools=[types.Tool(google_search=types.GoogleSearch())],
                temperature=0.0
            )
        )
        return response.text or "No recent news found."
    except Exception as e:
        return f"Search error: {str(e)}"


def get_evacuation_route(origin: str, destination: str) -> str:
    """
    Calculates driving evacuation routes, distance, travel time, and a Google Maps link between origin and destination.
    Args:
        origin (str): Starting location (e.g., "Tampa, FL").
        destination (str): Safe destination (e.g., "Orlando, FL").
    """
    api_key = os.environ.get("GOOGLE_MAPS_API_KEY", "")
    enc_orig = urllib.parse.quote(origin)
    enc_dest = urllib.parse.quote(destination)
    gmaps_url = f"https://www.google.com/maps/dir/?api=1&origin={enc_orig}&destination={enc_dest}&travelmode=driving"

    # 1. Official Google Maps API if key is provided
    if api_key and api_key != "YOUR_MAPS_API_KEY":
        try:
            resp = requests.get(
                "https://maps.googleapis.com/maps/api/directions/json",
                params={"origin": origin, "destination": destination, "key": api_key},
                timeout=10
            ).json()
            if resp.get("status") == "OK":
                leg = resp["routes"][0]["legs"][0]
                return (
                    f"Route ({origin} to {destination}): {leg['distance']['text']}, "
                    f"Est. Time: {leg['duration']['text']}. Google Maps: {gmaps_url}"
                )
        except Exception:
            pass

    # 2. Keyless OSRM + Geopy Fallback
    try:
        geolocator = Nominatim(user_agent="fema_poc_routing_tool")
        loc_o = geolocator.geocode(origin, timeout=10)
        loc_d = geolocator.geocode(destination, timeout=10)
        if loc_o and loc_d:
            osrm_url = (
                f"http://router.project-osrm.org/route/v1/driving/"
                f"{loc_o.longitude},{loc_o.latitude};{loc_d.longitude},{loc_d.latitude}"
                f"?overview=false&steps=true"
            )
            osrm_resp = requests.get(osrm_url, timeout=10).json()
            if osrm_resp.get("code") == "Ok":
                route = osrm_resp["routes"][0]
                miles = round(route["distance"] * 0.000621371, 1)
                mins = round(route["duration"] / 60)
                hrs, rem_mins = divmod(mins, 60)
                roads = []
                for s in route["legs"][0]["steps"]:
                    r = s.get("ref") or s.get("name")
                    if r and r not in roads:
                        roads.append(r)
                corridor = " ➔ ".join(roads[:7])
                return (
                    f"Evacuation Route ({origin} to {destination}):\n"
                    f"- Distance: {miles} miles\n"
                    f"- Estimated Time: {hrs} hrs {rem_mins} mins\n"
                    f"- Primary Corridor: {corridor}\n"
                    f"- Google Maps Link: {gmaps_url}"
                )
    except Exception:
        pass

    return f"Suggested Evacuation Route from {origin} to {destination}: View live directions at {gmaps_url}"

print("✅ All 3 core tools initialized!")

✅ All 3 core tools initialized!


Agents - Coordinator + Sub Agents and Validation

Set up with lower level models to prevent 429 errors

In [10]:
from google.adk import Agent, Workflow
from google.adk.workflow import START
from google.genai import types

# ---------------------------------------------------------
# 1. SPECIALIST AGENTS (Fulfilling Multi-Agent Requirement)
# ---------------------------------------------------------
weather_agent = Agent(
    name="weather_specialist",
    model="gemini-2.5-flash",
    description="Forecasts US weather and checks active NWS alerts.",
    instruction="Use `get_nws_weather` to retrieve weather forecasts and alerts for US cities.",
    tools=[get_nws_weather],
    before_model_callback=before_model_callback_handler,
    after_model_callback=after_model_callback_handler
)

news_search_agent = Agent(
    name="news_search_specialist",
    model="gemini-2.5-flash",
    description="Searches the internet for breaking disaster news and emergency alerts.",
    instruction="Use `search_disaster_news` to find current disaster news and situational awareness updates.",
    tools=[search_disaster_news],
    before_model_callback=before_model_callback_handler,
    after_model_callback=after_model_callback_handler
)

routing_agent = Agent(
    name="routing_specialist",
    model="gemini-2.5-flash",
    description="Provides evacuation routes to safety using Google Maps routing.",
    instruction="Use `get_evacuation_route` to provide driving distance, time, highway corridor, and the Google Maps link.",
    tools=[get_evacuation_route],
    before_model_callback=before_model_callback_handler,
    after_model_callback=after_model_callback_handler
)

qa_agent = Agent(
    name="preparedness_qa_specialist",
    model="gemini-2.5-flash",
    description="Answers emergency preparedness questions and explains agent capabilities.",
    instruction="Answer questions about emergency kits, disaster preparedness, and FEMA capabilities. Use `search_disaster_news` if needed.",
    tools=[search_disaster_news],
    before_model_callback=before_model_callback_handler,
    after_model_callback=after_model_callback_handler
)

# ---------------------------------------------------------
# 2. ROOT COORDINATOR AGENT
# ---------------------------------------------------------
fema_root_agent = Agent(
    name="fema_emergency_coordinator",
    model="gemini-2.5-flash",
    description="Root FEMA Coordinator that describes capabilities and coordinates weather, news, routing, and Q&A.",
    instruction="""
    You are the FEMA Emergency Response Coordinator (Current Year: 2026).

    CAPABILITIES:
    1. Real-time US Weather & Severe Weather Alerts (`get_nws_weather` / `weather_specialist`).
    2. Breaking Disaster News & Internet Search (`search_disaster_news` / `news_search_specialist`).
    3. Emergency Evacuation Routes & Google Maps Navigation (`get_evacuation_route` / `routing_specialist`).
    4. Emergency Preparedness Checklists & Safety Q&A (`preparedness_qa_specialist`).

    MISSION RULES:
    - Only assist with US weather, disasters, evacuation routes, emergency preparedness, and your capabilities.
    - Politely refuse off-topic requests (recipes, coding, sports, non-US travel).
    - Prefer using your direct tools (`get_nws_weather`, `search_disaster_news`, `get_evacuation_route`) to gather concrete facts in a single step and synthesize a complete draft response.
    """,
    tools=[get_nws_weather, search_disaster_news, get_evacuation_route],
    sub_agents=[weather_agent, news_search_agent, routing_agent, qa_agent],
    output_key="coordinator_draft",
    before_model_callback=before_model_callback_handler,
    after_model_callback=after_model_callback_handler,
    generate_content_config=types.GenerateContentConfig(temperature=0.2)
)

# ---------------------------------------------------------
# 3. SEQUENTIAL VALIDATION & REFINEMENT WORKFLOW (ADK 2.x)
# ---------------------------------------------------------
response_validator_agent = Agent(
    name="response_validator",
    model="gemini-2.5-flash",
    description="Validates draft responses for accuracy, safety, and mission alignment.",
    instruction="""
    Review the draft response.
    - If the draft is a guardrail refusal (e.g., refusing an off-topic request), keep the refusal brief and intact.
    - Otherwise, verify all weather facts, safety advice, highway corridors, and Google Maps URLs are preserved accurately.
    Output the verified response text for final refinement.
    """,
    output_key="validated_draft",
    before_model_callback=before_model_callback_handler,
    after_model_callback=after_model_callback_handler
)

response_refiner_agent = Agent(
    name="response_refiner",
    model="gemini-2.5-flash",
    description="Refines validated responses so they are clear, calm, well-written, and easy to understand.",
    instruction="""
    Refine the validated response for the public:
    - If it is a guardrail refusal, output the polite refusal directly in 1-2 sentences.
    - Otherwise, format the emergency information with clear headings and bullet points, using a calm, authoritative tone.
    - Preserve all specific numbers, forecasts, road names, and clickable Google Maps links.
    - Output ONLY the final polished message for the user.
    """,
    output_key="final_response",
    before_model_callback=before_model_callback_handler,
    after_model_callback=after_model_callback_handler
)

# Unified Sequential Workflow: Coordinator -> Validator -> Refiner
fema_system_workflow = Workflow(
    name="fema_emergency_response_system",
    description="End-to-end FEMA system that coordinates tools/agents and sequentially validates and refines every response.",
    edges=[
        (START, fema_root_agent, response_validator_agent, response_refiner_agent)
    ]
)

print("✅ FEMA Multi-Agent System & Sequential Validation Workflow ready!")

✅ FEMA Multi-Agent System & Sequential Validation Workflow ready!


Local Automated Test & and Interactive Chat

Includes Rate Limit Backoff to protect against 429 errors

In [11]:
import asyncio
from google.adk.runners import InMemoryRunner
from google.genai import types

fema_runner = InMemoryRunner(
    agent=fema_system_workflow,
    app_name="fema_poc_app"
)

session_id = "fema_session_1"
user_id = "fema_tester"

await fema_runner.session_service.create_session(
    app_name="fema_poc_app",
    user_id=user_id,
    session_id=session_id
)

# Local rolling history tracker for the pre-workflow guardrail check
rolling_history = []

async def ask_fema_agent(user_input: str, max_retries: int = 4) -> str:
    """Runs a query through the FEMA Workflow with multi-turn context and 429 backoff."""
    print("\n" + "=" * 70)
    print(f"🧑 USER: {user_input}")
    print("=" * 70)

    history_context_str = "\n".join(rolling_history[-6:])

    # Pre-check guardrail with rolling conversation history
    is_blocked, refusal_msg = validate_fema_input(user_input, history_context=history_context_str)
    if is_blocked:
        write_log_entry(f"[USER INPUT]: {user_input}")
        write_log_entry(f"[AGENT OUTPUT (BLOCKED)]: {refusal_msg}")
        print(f"🛑 [GUARDRAIL BLOCKED]: {refusal_msg}")
        return refusal_msg

    msg = types.Content(role="user", parts=[types.Part.from_text(text=user_input)])
    delay = 8

    for attempt in range(max_retries):
        try:
            final_text = ""
            async for event in fema_runner.run_async(
                user_id=user_id,
                session_id=session_id,
                new_message=msg
            ):
                if event.content and event.content.parts:
                    for part in event.content.parts:
                        if hasattr(part, "function_call") and part.function_call:
                            print(f"   🛠️ [TOOL CALL] ➔ {part.function_call.name}")
                        elif hasattr(part, "text") and part.text:
                            final_text = part.text
                elif hasattr(event, "output") and event.output:
                    final_text = str(event.output)

            # Update rolling history for follow-up turns
            rolling_history.append(f"User: {user_input}")
            rolling_history.append(f"FEMA Assistant: {final_text[:300]}")

            write_log_entry(f"[AGENT OUTPUT]: {final_text[:200]}...")
            print(f"\n🤖 FEMA ASSISTANT:\n{final_text}\n")
            return final_text

        except Exception as e:
            err_str = str(e)
            if "429" in err_str or "RESOURCE_EXHAUSTED" in err_str:
                if attempt < max_retries - 1:
                    print(f"⚠️ Quota limit (429) hit. Waiting {delay}s before retry {attempt+1}/{max_retries}...")
                    await asyncio.sleep(delay)
                    delay *= 2
                    continue
            print(f"❌ Execution error: {err_str}")
            return f"Error: {err_str}"

    return "Request failed after retries."

# Run automated test prompts covering all rubric requirements
test_prompts = [
    "Hello! What capabilities do you have, and what should I pack in a basic 72-hour emergency kit?",
    "There is flooding reported near Tampa, FL. What is the current weather in Tampa, FL, and what is the safest evacuation route to Orlando, FL?",
    "What are the latest news alerts or active severe weather advisories in the United States today?",
    "Can you write a Python script to scrape stock market prices and give me a chocolate cake recipe?"
]

for prompt in test_prompts:
    await ask_fema_agent(prompt)
    print("⏱️ Pacing 6 seconds between tests to respect RPM quota...")
    await asyncio.sleep(6)


🧑 USER: Hello! What capabilities do you have, and what should I pack in a basic 72-hour emergency kit?
2026-10-27 19:16:15 [INFO] [BEFORE_MODEL - fema_emergency_coordinator] (user): Hello! What capabilities do you have, and what should I pack in a basic 72-hour emergency kit?
2026-10-27 19:16:15 [INFO] [USER INPUT]: Hello! What capabilities do you have, and what should I pack in a basic 72-hour emergency kit?
2026-10-27 19:16:19 [INFO] [AFTER_MODEL - fema_emergency_coordinator]: Tool Call -> transfer_to_agent...
   🛠️ [TOOL CALL] ➔ transfer_to_agent
2026-10-27 19:16:19 [INFO] [BEFORE_MODEL - preparedness_qa_specialist] (user): For context: below is a transcript of what another agent did, quoted between <<<BEGIN_QUOTED_AGENT_CONTENT>>> and <<<END
2026-10-27 19:16:23 [INFO] [AFTER_MODEL - preparedness_qa_specialist]: Hello! As the preparedness_qa_specialist, I can answer your questions about emergency preparedness and explain the capab...
2026-10-27 19:16:23 [INFO] [BEFORE_MODEL - prepa

Interactive Test and Logs

In [12]:
import os
import asyncio

async def interactive_fema_session():
    print("=" * 70)
    print("🚨 FEMA EMERGENCY RESPONSE SYSTEM - INTERACTIVE TEST")
    print("Commands:")
    print("  - Type 'logs' to view the last 20 lines of the interaction log")
    print("  - Type 'full_logs' to print the entire log file")
    print("  - Type 'quit' or 'exit' to finish testing and proceed to deployment")
    print("=" * 70)

    while True:
        user_input = input("\nYou: ").strip()

        if not user_input:
            continue

        if user_input.lower() in ["quit", "exit"]:
            print("\nExiting interactive session. Ready for Cell 6 (Deployment)!")
            break

        # Command 1: View recent log entries
        if user_input.lower() == "logs":
            log_path = os.path.abspath(LOG_FILE)
            print(f"\n📁 Recent Logs ({log_path}):")
            print("-" * 70)
            if os.path.exists(log_path):
                with open(log_path, "r", encoding="utf-8") as f:
                    for line in f.readlines()[-20:]:
                        print(line.strip())
            else:
                print("Log file not found yet.")
            print("-" * 70)
            continue

        # Command 2: View the complete log file
        if user_input.lower() == "full_logs":
            log_path = os.path.abspath(LOG_FILE)
            print(f"\n📁 Full Log File ({log_path}):")
            print("=" * 70)
            if os.path.exists(log_path):
                with open(log_path, "r", encoding="utf-8") as f:
                    print(f.read())
            else:
                print("Log file not found yet.")
            print("=" * 70)
            continue

        # Send query through the rate-limit-protected FEMA workflow from Cell 5
        await ask_fema_agent(user_input)

        # Brief pause to keep your RPM/TPM quota healthy before your next prompt
        await asyncio.sleep(2)

# Start interactive testing
await interactive_fema_session()

🚨 FEMA EMERGENCY RESPONSE SYSTEM - INTERACTIVE TEST
Commands:
  - Type 'logs' to view the last 20 lines of the interaction log
  - Type 'full_logs' to print the entire log file
  - Type 'quit' or 'exit' to finish testing and proceed to deployment

You: hey i live in Easton, WA and it is snowing outside should i be worried?

🧑 USER: hey i live in Easton, WA and it is snowing outside should i be worried?
2026-10-27 19:22:04 [INFO] [BEFORE_MODEL - fema_emergency_coordinator] (user): hey i live in Easton, WA and it is snowing outside should i be worried?
2026-10-27 19:22:04 [INFO] [USER INPUT]: hey i live in Easton, WA and it is snowing outside should i be worried?
2026-10-27 19:22:06 [INFO] [AFTER_MODEL - fema_emergency_coordinator]: Tool Call -> get_nws_weather...
   🛠️ [TOOL CALL] ➔ get_nws_weather
2026-10-27 19:22:07 [INFO] [BEFORE_MODEL - fema_emergency_coordinator] (user): [Tool/System Event]
2026-10-27 19:22:09 [INFO] [AFTER_MODEL - fema_emergency_coordinator]: Yes, you should be aw

Deploy Agents

In [13]:
import os
import vertexai
from vertexai import agent_engines
from vertexai.agent_engines import AdkApp

project_id = os.environ["GOOGLE_CLOUD_PROJECT"]
location = os.environ["GOOGLE_CLOUD_LOCATION"]
staging_bucket = f"gs://{project_id}-agent-staging"

# 1. Create staging bucket if it doesn't exist yet
!gcloud storage buckets create {staging_bucket} --project={project_id} --location={location} --quiet 2>/dev/null || true

# 2. Initialize Vertex AI
vertexai.init(
    project=project_id,
    location=location,
    staging_bucket=staging_bucket
)

# 3. Wrap the FEMA Agent/Workflow in AdkApp
fema_app = AdkApp(
    agent=fema_system_workflow,
    enable_tracing=True
)

print("🚀 Deploying FEMA Emergency Response System to Agent Platform (takes ~3-5 minutes)...")

# 4. Deploy with complete ADK + Tool requirements
remote_fema_app = agent_engines.create(
    display_name="FEMA Emergency Response System",
    agent_engine=fema_app,
    requirements=[
        "google-cloud-aiplatform[agent_engines,adk]>=1.112",
        "google-adk",
        "google-genai>=2.12.1",
        "requests>=2.31.0",
        "geopy>=2.4.1"
    ]
)

print(f"✅ Deployment Successful! Resource Name: {remote_fema_app.resource_name}")

# 5. Test the Deployed Remote Agent on Agent Platform
print("\n--- Testing Deployed Remote Agent ---")
for event in remote_fema_app.stream_query(
    user_id="cloud_tester",
    message="What is the current weather in Seattle, WA and how do I evacuate to Spokane, WA?"
):
    print(event)

INFO:vertexai.agent_engines:Identified the following requirements: {'google-cloud-aiplatform': '2.6.0', 'pydantic': '2.13.4', 'cloudpickle': '3.1.2'}
INFO:vertexai.agent_engines:The following requirements are appended: {'pydantic==2.13.4', 'cloudpickle==3.1.2'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]>=1.112', 'google-adk', 'google-genai>=2.12.1', 'requests>=2.31.0', 'geopy>=2.4.1', 'pydantic==2.13.4', 'cloudpickle==3.1.2']


🚀 Deploying FEMA Emergency Response System to Agent Platform (takes ~3-5 minutes)...


INFO:vertexai.agent_engines:Using bucket qwiklabs-gcp-04-c28c99d58f4c-agent-staging
INFO:vertexai.agent_engines:Wrote to gs://qwiklabs-gcp-04-c28c99d58f4c-agent-staging/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-04-c28c99d58f4c-agent-staging/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-04-c28c99d58f4c-agent-staging/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine backing LRO: projects/253953783726/locations/us-central1/reasoningEngines/3341550252107759616/operations/7804976417543815168
INFO:vertexai.agent_engines:View progress and logs at https://console.cloud.google.com/logs/query?project=qwiklabs-gcp-04-c28c99d58f4c
INFO:vertexai.agent_engines:AgentEngine created. Resource name: projects/253953783726/locations/us-central1/reasoningEngines/3341550252

✅ Deployment Successful! Resource Name: projects/253953783726/locations/us-central1/reasoningEngines/3341550252107759616

--- Testing Deployed Remote Agent ---
{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'function_call': {'id': 'adk-d748ae55-5215-459c-a635-1206ca3fd516', 'args': {'city': 'Seattle, WA'}, 'name': 'get_nws_weather'}, 'thought_signature': 'AY89a18hyBDgVMlPnANprdLx5FeWblzqZtg9yvlZ1Vx4EuTwnjbEshEKdkLUB7VevakwPl9jwKlCiQcnbY6O9-ZOfTGjK9O8fXoDpm1_pWBjC1tcv_Did5JATZBK7WGn6V7RNoUA9wCnsGeFDpfsZ2toK-HaGXgzM2xPFkHzhotpGIdO0QUQY160WeWzK5NeG9lzkpboz2U4YbrgNYGv5Dx569IrV2LteUZf754esY7GzrMEfVm6z2jdbtS9Tuxjs2G1F1hDcxFZEWwdxqbkvVS-ln7wNMx5xytY8abYGtTM-Ehb7uinhzJeTXDqgXogV8THtpYUWBoSz_FXRfEihLsC5iXBj7vW06HWtqAKUQ6yyMNnhPV0fFPCFniNKN6530nd0n4ovNQtCtzaFQXfIszTAwDmHZUg5SN5s9kh7RZgJJRQsBD8p0ezqQf86Cl8TU_fTED21yImOykf9qrNSXMAPHHxJHaP5zPWkSj651nmYHpx00T9zScPcL1_LtphPLGedi8se7jYLjjIzNpkoqShCUa1aZICRnAMQ08zq5Sr03Q77kWIVMu2XaOn2uwlJ_juUhfl6YOHvy9GEsigFCbqYA1UOwnYAoggI7OpnMAOK-zy11Hp